# Anomaly Detection and Clustering

**Objective:** Identify unusual monetary weeks, cluster similar balance-sheet regimes, and reduce dimensions with PCA.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rbi_utils import *

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 120)


In [ ]:
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

df = load_cleaned()
features = KEY_SERIES
model_df = df.dropna(subset=features).copy()
X = StandardScaler().fit_transform(model_df[features])

In [ ]:
model_df['zscore_anomaly'] = model_df['reserve_money_zscore'].abs() > 3
model_df['isolation_forest_anomaly'] = IsolationForest(contamination=0.03, random_state=42).fit_predict(X) == -1
model_df['lof_anomaly'] = LocalOutlierFactor(contamination=0.03).fit_predict(X) == -1
model_df.loc[model_df[['zscore_anomaly','isolation_forest_anomaly','lof_anomaly']].any(axis=1), ['date','reserve_money','zscore_anomaly','isolation_forest_anomaly','lof_anomaly']].head(20)

In [ ]:
model_df['kmeans_cluster'] = KMeans(n_clusters=3, random_state=42, n_init=20).fit_predict(X)
model_df.groupby('kmeans_cluster')[features].mean()

In [ ]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X)
model_df['pc1'] = coords[:, 0]
model_df['pc2'] = coords[:, 1]
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=model_df, x='pc1', y='pc2', hue='kmeans_cluster', palette='tab10', ax=ax)
ax.set_title('PCA View of Monetary Conditions')
save_figure(fig, '07_pca_clusters.png')